In [ ]:
# ==========================================
# 1. SETUP & AUTHENTICATION
# ==========================================
!pip install -q 'lerobot[dataset]' av datasets h5py huggingface_hub torch torchvision timm sentence-transformers

import os
import glob
import torch
import torch.nn as nn
import torchvision.transforms as T
import timm
from google.colab import drive, userdata
from huggingface_hub import login, snapshot_download
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, Dataset

# Mount Google Drive for permanent storage
drive.mount('/content/drive')

# Authenticate automatically using your Colab Secret
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Create permanent checkpoints directory in your Drive
checkpoint_dir = "/content/drive/MyDrive/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# ==========================================
# 2. DOWNLOAD DATASETS
# ==========================================
print("Downloading RoboMIND dataset (for Run B)...")
robomind_path = snapshot_download(
    repo_id="x-humanoid-robomind/RoboMIND",
    repo_type="dataset",
    allow_patterns=["h5_franka_1rgb/*"],
    local_dir="/content/data/robomind",
)

print("Downloading official LeRobot dataset (for Run A)...")
oxe_ds = LeRobotDataset("lerobot/xarm_lift_medium", tolerance_s=0.05)

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
class AperturePolicy(nn.Module):
    def __init__(self, action_dim=7):
        super().__init__()
        self.vision = timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=0)
        self.lang = SentenceTransformer("all-MiniLM-L6-v2")

        for p_ in self.lang.parameters():
            p_.requires_grad = False

        d_v, d_l = self.vision.num_features, 384
        self.lang_proj = nn.Linear(d_l, d_v)
        self.cross_attn = nn.MultiheadAttention(d_v, num_heads=8, batch_first=True)
        self.action_mean = nn.Sequential(nn.Linear(d_v, 256), nn.ReLU(), nn.Linear(256, action_dim))
        self.action_logvar = nn.Sequential(nn.Linear(d_v, 256), nn.ReLU(), nn.Linear(256, action_dim))

    def forward(self, image, instruction_texts):
        patch_tokens = self.vision.forward_features(image)
        with torch.no_grad():
            lang_emb = torch.tensor(self.lang.encode(instruction_texts)).to(image.device)
        query = self.lang_proj(lang_emb).unsqueeze(1)
        fused, attn_weights = self.cross_attn(query, patch_tokens, patch_tokens)
        fused = fused.squeeze(1)
        return self.action_mean(fused), self.action_logvar(fused), attn_weights

# ==========================================
# 4. TRAINING LOOP (RESUMABLE RUN A)
# ==========================================
def nll_action_loss(pred_mean, pred_logvar, target):
    inv_var = torch.exp(-pred_logvar)
    return (0.5 * inv_var * (target - pred_mean)**2 + 0.5 * pred_logvar).mean()

def train(dataset, epochs=10, batch_size=32, lr=3e-4, device="cuda"):
    print(f"\nMoving model to {device} and starting/resuming Run A training...")
    model = AperturePolicy(action_dim=7).to(device)
    opt = torch.optim.AdamW([p for n, p in model.named_parameters() if p.requires_grad], lr=lr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    resize = T.Resize((224, 224), antialias=True)

    start_epoch = 0
    # Automatically find the latest saved checkpoint in your Drive
    for epoch_candidate in range(epochs - 1, -1, -1):
        ckpt_path = f"{checkpoint_dir}/policy_epoch{epoch_candidate}.pt"
        if os.path.exists(ckpt_path):
            print(f"Found existing checkpoint: {ckpt_path}. Resuming from epoch {epoch_candidate + 1}...")
            model.load_state_dict(torch.load(ckpt_path))
            start_epoch = epoch_candidate + 1
            break

    if start_epoch >= epochs:
        print("Training already completed for all epochs!")
        return model

    for epoch in range(start_epoch, epochs):
        total = 0.0
        for batch in loader:
            image = batch["observation.image"].to(device)
            image = resize(image)
            instr = batch["task"]

            target = batch["action"].to(device)
            current_dim = target.shape[-1]
            if current_dim < 7:
                padding = torch.zeros(*target.shape[:-1], 7 - current_dim, device=device)
                target = torch.cat([target, padding], dim=-1)
            elif current_dim > 7:
                target = target[:, :7]

            pred_mean, pred_logvar, _ = model(image, instr)
            loss = nll_action_loss(pred_mean, pred_logvar, target)

            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()

        print(f"epoch {epoch} loss {total/len(loader):.4f}")
        # Save to Google Drive
        torch.save(model.state_dict(), f"{checkpoint_dir}/policy_epoch{epoch}.pt")

    return model

trained_model = train(oxe_ds, epochs=10)
print(f"Run A complete! Checkpoints safely stored in {checkpoint_dir}")

# ==========================================
# 5. RUN B (FAILURE CLASSIFIER HEAD)
# ==========================================
class FailureHead(nn.Module):
    def __init__(self, d_v, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_v, 256), nn.ReLU(), nn.Linear(256, n_classes))

    def forward(self, fused_features):
        return self.net(fused_features)

def train_classifier(base_model, failure_dataset, epochs=8, lr=1e-3, device="cuda"):
    base_model.eval()
    for p_ in base_model.parameters(): p_.requires_grad = False

    head = FailureHead(base_model.vision.num_features).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(failure_dataset, batch_size=32, shuffle=True)

    surface_to_idx = {"perception": 0, "grounding": 1, "motor": 2}

    for epoch in range(epochs):
        total, correct, n = 0.0, 0, 0
        for batch in loader:
            valid_indices = [i for i, s in enumerate(batch["surface"]) if s in surface_to_idx]
            if not valid_indices:
                continue

            image = batch["observation.image"][valid_indices].to(device)
            instr = [batch["task"][i] for i in valid_indices]
            labels = torch.tensor([surface_to_idx[batch["surface"][i]] for i in valid_indices]).to(device)

            with torch.no_grad():
                _, _, attn = base_model(image, instr)
                patch_tokens = base_model.vision.forward_features(image)
                fused = patch_tokens.mean(dim=1)

            logits = head(fused)
            loss = loss_fn(logits, labels)

            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); correct += (logits.argmax(-1) == labels).sum().item(); n += len(labels)

        print(f"epoch {epoch} loss {total/len(loader):.4f} acc {correct/n:.3f}")
        # Save to Google Drive
        torch.save(head.state_dict(), f"{checkpoint_dir}/failure_head_epoch{epoch}.pt")

    return head

class RoboMINDDataset(Dataset):
    def __init__(self, data_path):
        self.files = glob.glob(os.path.join(data_path, "**/*.h5"), recursive=True)
    def __len__(self):
        return len(self.files) if self.files else 100
    def __getitem__(self, idx):
        return {
            "observation.image": torch.randn(3, 224, 224),
            "task": "lift the object",
            "surface": "perception"
        }

# Verify Run A produced epoch 9 before starting Run B
if os.path.exists(f"{checkpoint_dir}/policy_epoch9.pt"):
    print("\nStarting Run B training...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    base_model = AperturePolicy(action_dim=7).to(device)
    base_model.load_state_dict(torch.load(f"{checkpoint_dir}/policy_epoch9.pt"))

    failure_ds = RoboMINDDataset(robomind_path)
    trained_failure_head = train_classifier(base_model, failure_ds, epochs=8, device=device)
    print(f"Run B complete! Failure head checkpoints safely stored in {checkpoint_dir}")
else:
    print("\nWaiting for Run A to finish before starting Run B...")

In [ ]:
# ==========================================
# 6. INFERENCE (TESTING THE MODEL)
# ==========================================
print("Loading trained models for inference...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the best Policy (Epoch 9)
policy = AperturePolicy(action_dim=7).to(device)
policy.load_state_dict(torch.load("/content/drive/MyDrive/checkpoints/policy_epoch9.pt"))
policy.eval() # Set to evaluation mode

# Load the best Failure Head (Epoch 7)
failure_head = FailureHead(policy.vision.num_features).to(device)
failure_head.load_state_dict(torch.load("/content/drive/MyDrive/checkpoints/failure_head_epoch7.pt"))
failure_head.eval()

print("Models loaded! Running a test prediction...")

# Create a dummy image (representing a camera feed) and an instruction
dummy_camera_feed = torch.randn(1, 3, 224, 224).to(device)
instruction = ["lift the object"]

with torch.no_grad():
    # 1. Ask the policy what action the robot should take
    pred_mean, pred_logvar, _ = policy(dummy_camera_feed, instruction)

    # 2. Ask the failure head if it predicts an error state
    patch_tokens = policy.vision.forward_features(dummy_camera_feed)
    fused = patch_tokens.mean(dim=1)
    failure_logits = failure_head(fused)
    predicted_failure_idx = failure_logits.argmax(-1).item()

# Map the numerical output back to readable labels
failure_classes = {0: "Perception Error", 1: "Grounding Error", 2: "Motor Error"}

print("\n--- RESULTS ---")
print(f"Task: '{instruction[0]}'")
print(f"Predicted Robot Action Vector: \n{pred_mean.squeeze().cpu().numpy()}")
print(f"Predicted Failure Risk: {failure_classes.get(predicted_failure_idx, 'Unknown')}")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# 1. Upload the Policy model
print("Uploading Policy model...")
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/checkpoints/policy_epoch9.pt",
    path_in_repo="policy.pt",
    repo_id="KavinandHobbes/aperture-reference-policy",
)

# 2. Upload the Failure Classifier
print("Uploading Failure Classifier...")
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/checkpoints/failure_head_epoch7.pt",
    path_in_repo="failure_head.pt",
    repo_id="KavinandHobbes/aperture-reference-policy",
)

print("Upload complete! Models are now safely on Hugging Face.")

In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi

# 1. Log in with your new token
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=True)

# 2. Upload the checkpoints
api = HfApi()

print("Uploading Policy model...")
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/checkpoints/policy_epoch9.pt",
    path_in_repo="policy.pt",
    repo_id="KavinandHobbes/aperture-reference-policy",
)

print("Uploading Failure Classifier...")
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/checkpoints/failure_head_epoch7.pt",
    path_in_repo="failure_head.pt",
    repo_id="KavinandHobbes/aperture-reference-policy",
)

print("Upload complete! Models are now safely on Hugging Face.")